In [ ]:
# libraries for data loading and system/path settings
import json
import sys
from pathlib import Path

# libraries for model building and training
import torch
from torch.utils.data import Dataset

# set path to project root and import custom classes and functions
base_path = Path.cwd() / "../../../"
sys.path.append(str(base_path.resolve()))
from utils.evaluation import cv_stance

In [ ]:
# create the dataset class
class StanceDataset(Dataset):
    def __init__(self, data, tokenizer, label2id, max_len=128):
        self.dataset = []
        for item in data:
            sentence = item["sentence"]
            target = item["group"]
            stance = item["stance"]
            label = label2id[stance]
            
            encoded = tokenizer(
                sentence,
                target,
                truncation=True,
                padding="max_length",
                max_length=max_len,
                return_tensors="pt"
                )
            self.dataset.append({
                "input_ids": encoded["input_ids"].squeeze(0),
                "attention_mask": encoded["attention_mask"].squeeze(0),
                "label": torch.tensor(label, dtype=torch.long)
                })
    
    def __len__(self):
        return len(self.dataset)
    
    def __getitem__(self, idx):
        return self.dataset[idx]

In [ ]:
# load training and validation data
with open("../../../01_data/training_validation_sets/stance/training_set.json", "r") as f:
    data = json.load(f)

# initialize dictionary for the sentiment classes
sent_dict = set()

# loop through all sentences
for task in data:
    sent_dict.add(task[0]["stance"])

# sort the tag dictionary
label_list = sorted(sent_dict)

# dictionaries that convert from id to tag and vice versa
label_to_id = {tag: i for i, tag in enumerate(label_list)}
id_to_label = {id: label for label, id in label_to_id.items()}

# load the results from hyperparameter tuning
with open("hyperparameter_tuning_results/hyperparametertuning_results.json", "r") as f:
    hyperparameter_tuning_results = json.load(f)

In [ ]:
# set all model names
model_names = ["roberta-base", "bert-base-cased", "distilbert-base-cased", "microsoft/deberta-v3-base"]

# apply cross validation to all models with their best hyperparameters
average_metrics = cv_stance(model_names=model_names, training_data=data, dataset_class=StanceDataset, label2id=label_to_id,
                            id2label=id_to_label, num_folds=5, optimal_configurations=hyperparameter_tuning_results, seed=3)

# export the cross-validation metrics
with open("../cross_val_results/bert_sequence.json", "w") as f:
    json.dump(average_metrics, f, indent=4)